# Load category distribution information
the statistic information about arXiv is stored in `category_count./csv`

In [4]:
import csv


def load_csv(file):
    data = []
    with open(file, newline='') as csvfile:
        cr = csv.DictReader(csvfile)
        for row in cr:
            data.append(row)

    return data


file = 'data/category_count.csv'
category_data = load_csv(file)

# Download source files of papers by 

In [5]:
import arxiv
import os
from typing import List, Dict


def arxiv_download(data: List[Dict], path: str):
    papers = set()

    for row in data:
        category, count = row["categories"], int(row["count"])
        sub_directory = os.path.join(path, category)
        os.makedirs(sub_directory, exist_ok=True)

        if len(os.listdir(sub_directory)) > count:
            print(f"Complete process for {category}, {count} papers")
            continue

        search = arxiv.Search(
            query=category,
            max_results=count,
            sort_by=arxiv.SortCriterion.SubmittedDate,
        )

        for result in search.results():
            if len(os.listdir(sub_directory)) > count:
                break

            if result.entry_id in papers:
                continue
            papers.add(result.entry_id)
            result.download_source(dirpath=sub_directory)

        print(f"Complete process for {category}, {count} papers")

target_directory = os.path.expanduser("~/vrdu_data")
arxiv_download(category_data, target_directory)

# Process
After downloading the source files of PDFs, we first unzip them, then we run the scripts to generate the annotation result, finaly, we export the result

## 1. unzip the arXiv source files

In [6]:
import os
import tarfile


def extract_all_tar_gz(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".tar.gz"):
                file_path = os.path.join(root, file)
                extract_path = os.path.splitext(file_path)[0]
                extract_tar_gz(file_path, extract_path)


def extract_tar_gz(file_path, extract_path):
    with tarfile.open(file_path, 'r:gz') as tar:
        tar.extractall(extract_path)


extract_all_tar_gz(target_directory)

In [ ]:
# find all tex files
import os


def find_all_tex_files(directory) -> List[str]:
    tex_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            # skip the _rendered.tex files
            if file.endswith(".tex") and not file.endswith("_rendered.tex"):
                file_path = os.path.join(root, file)
                tex_files.append(file_path)
    return tex_files


tex_files = find_all_tex_files(target_directory)
print(f"{len(tex_files)} tex files are found in {target_directory}")

# Preprocess

1. Only process the main latex file in the directory

In [ ]:
def preprocess_tex_files(tex_files: List[str]) -> List[str]:
    result = []
    for tex_file in tex_files:
        print(f"processing {tex_file}")
        try:
            tex_text = open(tex_file).read()
            # check if it is a main tex file
            if tex_text.find(r"\begin{document}") == -1:
                continue
            result.append(tex_file)
        except:
            pass

    return result


print(f"before processing, there are {len(tex_files)} tex files")
tex_files = preprocess_tex_files(tex_files)
print(f"after processing, there are {len(tex_files)} tex files")

In [ ]:
# run vrdu_data process for all files:
import tqdm
import subprocess


def run_script(tex_files, script_path, time_out=300) -> None:
    for tex_file in tqdm.tqdm(tex_files):
        print(f"processing {tex_file}")
        try:
            # Execute the shell script for each object
            subprocess.run(['bash', script_path, tex_file], check=True,
                        timeout=time_out)
            print(f"Script executed successfully for {tex_file}")
        except subprocess.CalledProcessError as e:
            # Handle any exceptions that occur during script execution
            print(f"Script failed for {tex_file}: {e}")
        except subprocess.TimeoutExpired as e:
            # Handle any exceptions that occur during script execution
            print(f"Script timed out for {tex_file}: {e}")
        else:
            # Script executed successfully
            print(f"Script executed successfully for {tex_file}")

script_path = os.path.expanduser("./pipeline.sh")
run_script(tex_files, script_path)

In [7]:
import os
import shutil
import zipfile


def extract_result(source_directory, destination_directory):
    # Create the destination directory if it doesn't exist
    os.makedirs(destination_directory, exist_ok=True)

    # Iterate over the nested directories
    for root, dirs, files in os.walk(source_directory):
        if "output" in dirs and "result" in os.listdir(os.path.join(root, "output")):
            # Construct the path to the "result" subdirectory
            result_directory = os.path.join(root, "output", "result")

            # Create a new directory in the destination directory
            new_directory = os.path.join(
                destination_directory, os.path.basename(root))
            os.makedirs(new_directory, exist_ok=True)

            # Copy the "result" subdirectory to the new directory
            shutil.copytree(
                result_directory, os.path.join(
                    new_directory, os.path.basename(root))
            )

            # Zip the new directory
            zip_file_path = os.path.join(
                destination_directory, f"{os.path.basename(root)}.zip"
            )
            with zipfile.ZipFile(zip_file_path, "w", zipfile.ZIP_DEFLATED) as zip_file:
                for folder_name, _, file_names in os.walk(new_directory):
                    for file_name in file_names:
                        file_path = os.path.join(folder_name, file_name)
                        zip_file.write(
                            file_path, os.path.relpath(
                                file_path, new_directory)
                        )

            # Remove the copied "result" subdirectory
            shutil.rmtree(new_directory)


destination_directory = os.path.expanduser("~/vrdu_data/annotations")
# os.mkdir(destination_directory)
extract_result(sub_directory, destination_directory)